# 03 — IA : données, cibles et références (S3-07)

Cette étape prépare la prévision de volatilité sans entraîner de LSTM et sans importer TensorFlow. Les cours AAPL sont demandés sur 10 ans avec `yfinance`; le fichier local `data/aapl_10y_fallback.csv` prend automatiquement le relais hors connexion.

Deux jeux chronologiques suivent `CLAUDE.md` §9.5 S4-03 :

- **V0 (guide)** : 30 volatilités réalisées glissantes sur 5 jours pour prévoir celle du lendemain. La cible partage quatre rendements avec la dernière entrée, ce qui favorise la persistance.
- **V1 (rigoureuse)** : 30 rendements journaliers pour prévoir la volatilité annualisée des 5 jours suivants. Les rendements de la cible ne chevauchent jamais la fenêtre d'entrée.

Le découpage est chronologique 70 / 15 / 15. Les normalisations sont apprises exclusivement sur le train. Les références préparées sont la persistance et l'EWMA RiskMetrics, définie par $h_t = 0{,}94 h_{t-1} + 0{,}06 r_t^2$ puis annualisée par $\sqrt{252}$.

In [1]:
from pathlib import Path
import sys, tempfile
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import yfinance as yf

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from data import load_prices_info, log_returns, rolling_realized_vol
yf.set_tz_cache_location(str(Path(tempfile.gettempdir()) / 'yfinance_cache'))

CSV_FALLBACK = ROOT / 'data' / 'aapl_10y_fallback.csv'
prices, SOURCE = load_prices_info('AAPL', period='10y', csv_fallback=CSV_FALLBACK)
prices = prices.astype(float).sort_index()
prices.index = pd.to_datetime(prices.index).tz_localize(None)
prices.index.name, prices.name = 'Date', 'Close'

assert len(prices) >= 10 * 240, 'Historique trop court pour dix ans de bourse.'
assert prices.index.is_monotonic_increasing and prices.index.is_unique
assert prices.notna().all() and np.isfinite(prices).all() and (prices > 0).all()
returns = log_returns(prices)
assert returns.notna().all() and np.isfinite(returns).all()
print(f'SOURCE = {SOURCE} | {len(prices)} cours | {prices.index.min().date()} → {prices.index.max().date()}')
print(f'Colonnes CSV attendues : Date, Close | valeurs manquantes : {int(prices.isna().sum())}')

SOURCE = yfinance | 2514 cours | 2016-11-21 → 2026-11-20
Colonnes CSV attendues : Date, Close | valeurs manquantes : 0


In [2]:
TRADING_DAYS, WINDOW, HORIZON = 252, 30, 5

def make_v0(r, window=WINDOW, vol_window=HORIZON):
    vol = rolling_realized_vol(r, window=vol_window).dropna()
    X = np.stack([vol.iloc[i:i + window].to_numpy() for i in range(len(vol) - window)])
    y = vol.iloc[window:].to_numpy()
    dates = vol.index[window:]
    return X[..., None], y, dates

def make_v1(r, window=WINDOW, horizon=HORIZON):
    values = r.to_numpy(dtype=float)
    n = len(values) - window - horizon + 1
    X = np.stack([values[i:i + window] for i in range(n)])
    y = np.array([values[i + window:i + window + horizon].std(ddof=1) * np.sqrt(TRADING_DAYS) for i in range(n)])
    input_end = r.index[window - 1:window - 1 + n]
    target_start = r.index[window:window + n]
    target_end = r.index[window + horizon - 1:window + horizon - 1 + n]
    assert (target_start > input_end).all(), 'Fuite détectée entre entrée et cible V1.'
    return X[..., None], y, pd.DataFrame({'input_end': input_end, 'target_start': target_start, 'target_end': target_end})

def prepare_chronological(X, y):
    train_end, val_end = int(0.70 * len(y)), int(0.85 * len(y))
    x_scaler, y_scaler = MinMaxScaler(), MinMaxScaler()
    x_scaler.fit(X[:train_end].reshape(-1, X.shape[-1]))
    y_scaler.fit(y[:train_end, None])
    X_scaled = x_scaler.transform(X.reshape(-1, X.shape[-1])).reshape(X.shape)
    y_scaled = y_scaler.transform(y[:, None]).ravel()
    return {'X': X, 'y': y, 'X_scaled': X_scaled, 'y_scaled': y_scaled,
            'train_end': train_end, 'val_end': val_end, 'x_scaler': x_scaler, 'y_scaler': y_scaler}

X0, y0, dates0 = make_v0(returns)
X1, y1, dates1 = make_v1(returns)
datasets = {'V0': prepare_chronological(X0, y0), 'V1': prepare_chronological(X1, y1)}
for name, d in datasets.items():
    a, b, n = d['train_end'], d['val_end'], len(d['y'])
    assert 0 < a < b < n
    print(f'{name} : X={d["X"].shape}, y={d["y"].shape} | train={a}, validation={b-a}, test={n-b}')

V0 : X=(2479, 30, 1), y=(2479,) | train=1735, validation=372, test=372
V1 : X=(2479, 30, 1), y=(2479,) | train=1735, validation=372, test=372


In [3]:
ALPHA_EWMA = 0.94  # lambda RiskMetrics dans CLAUDE.md

def persistence_v0(X):
    return X[:, -1, 0]

def persistence_v1(X, horizon=HORIZON):
    return X[:, -horizon:, 0].std(axis=1, ddof=1) * np.sqrt(TRADING_DAYS)

def ewma_v1(X, alpha=ALPHA_EWMA):
    # Même définition que le projet : h_t = alpha*h_(t-1) + (1-alpha)*r_t².
    r = X[:, :, 0]
    variance = r.var(axis=1, ddof=1)
    for t in range(r.shape[1]):
        variance = alpha * variance + (1.0 - alpha) * r[:, t] ** 2
    return np.sqrt(variance * TRADING_DAYS)

d0, d1 = datasets['V0'], datasets['V1']
X0_test = d0['X'][d0['val_end']:]
X1_test = d1['X'][d1['val_end']:]
references = {
    'V0_persistance': persistence_v0(X0_test),
    'V1_persistance': persistence_v1(X1_test),
    'V1_EWMA_0.94': ewma_v1(X1_test),
}
for name, prediction in references.items():
    assert len(prediction) > 0 and np.isfinite(prediction).all() and (prediction >= 0).all()
    print(f'{name} : {len(prediction)} prévisions, moyenne={prediction.mean():.2%}')
print('Préparation S3-07 terminée : aucun LSTM entraîné, aucun TensorFlow utilisé.')

V0_persistance : 372 prévisions, moyenne=21.24%
V1_persistance : 372 prévisions, moyenne=21.41%
V1_EWMA_0.94 : 372 prévisions, moyenne=23.65%
Préparation S3-07 terminée : aucun LSTM entraîné, aucun TensorFlow utilisé.
